# 🔑 Solutions: Lesson 11 · Recurrent Networks: RNN, LSTM & GRU

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 11.1: Vanilla RNN, forward pass  &nbsp; `⭐⭐`

Implement `rnn_forward(x_seq, Wxh, Whh, b)` returning all hidden states `(T, n_hidden)` with $h_0=0$ and $h_t=\tanh(x_tW_{xh}+h_{t-1}W_{hh}+b)$. The check compares with PyTorch's `nn.RNN`.


In [2]:
def rnn_forward(x_seq, Wxh, Whh, b):
    h, states = np.zeros(Whh.shape[0]), []
    for x_t in x_seq:
        h = np.tanh(x_t @ Wxh + h @ Whh + b)
        states.append(h)
    return np.array(states)

In [3]:
# ▶ self-check
torch.manual_seed(1)
rnn = nn.RNN(3, 4, batch_first=True)
x = torch.randn(1, 6, 3)
ref, _ = rnn(x)
p = {k: v.detach().numpy() for k, v in rnn.named_parameters()}
mine = rnn_forward(x[0].numpy(), p["weight_ih_l0"].T, p["weight_hh_l0"].T, p["bias_ih_l0"] + p["bias_hh_l0"])
dl.check("hidden states match nn.RNN", mine is not None and mine.shape == (6, 4) and np.allclose(mine, ref[0].detach().numpy(), atol=1e-6))

✅ hidden states match nn.RNN


---
### 🧩 Problem 11.2: Count recurrent parameters  &nbsp; `⭐`

Write `rnn_params(i, h)`, `gru_params(i, h)` and `lstm_params(i, h)` using the PyTorch convention (two bias vectors).


In [4]:
def rnn_params(i, h):
    return h * i + h * h + 2 * h
def gru_params(i, h):
    return 3 * rnn_params(i, h)
def lstm_params(i, h):
    return 4 * rnn_params(i, h)

In [5]:
# ▶ self-check
for name, f, mod in [("RNN", rnn_params, nn.RNN), ("GRU", gru_params, nn.GRU), ("LSTM", lstm_params, nn.LSTM)]:
    v = f(7, 16)
    dl.check(f"{name} formula equals PyTorch", v is not None and v == dl.count_params(mod(7, 16)))

✅ RNN formula equals PyTorch
✅ GRU formula equals PyTorch
✅ LSTM formula equals PyTorch


---
### 🧩 Problem 11.3: A GRU cell from scratch  &nbsp; `⭐⭐⭐`

Implement `gru_cell(x, h, W_ih, W_hh, b_ih, b_hh)` (NumPy, shapes as in PyTorch: `W_ih (3H, n_in)`, `W_hh (3H, H)`, biases `(3H,)`; gate order **r, z, n**).
Use $r=\sigma(\dots),\ z=\sigma(\dots),\ n=\tanh(W_{in}x+b_{in}+r\odot(W_{hn}h+b_{hn})),\ h'=(1-z)\odot n+z\odot h$. It must match `nn.GRUCell`.


In [6]:
def gru_cell(x, h, W_ih, W_hh, b_ih, b_hh):
    sig = lambda a: 1 / (1 + np.exp(-a))
    gi, gh = x @ W_ih.T + b_ih, h @ W_hh.T + b_hh
    i_r, i_z, i_n = np.split(gi, 3, axis=-1)
    h_r, h_z, h_n = np.split(gh, 3, axis=-1)
    r, z = sig(i_r + h_r), sig(i_z + h_z)
    n = np.tanh(i_n + r * h_n)
    return (1 - z) * n + z * h

In [7]:
# ▶ self-check
torch.manual_seed(2)
cell = nn.GRUCell(3, 5); x = torch.randn(4, 3); h = torch.randn(4, 5)
ref = cell(x, h).detach().numpy()
p = {k: v.detach().numpy() for k, v in cell.named_parameters()}
mine = gru_cell(x.numpy(), h.numpy(), p["weight_ih"], p["weight_hh"], p["bias_ih"], p["bias_hh"])
dl.check("matches nn.GRUCell", mine is not None and np.allclose(mine, ref, atol=1e-6))

✅ matches nn.GRUCell


**Why this works.** A GRU is just two sigmoid gates and one tanh candidate: the update gate `z` interpolates between keeping the old state and adopting the new candidate.

---
### 🧩 Problem 11.4: Sliding windows  &nbsp; `⭐⭐`

Write `make_windows(series, window, horizon=1)` for a 1-D NumPy array. Return `(X, y)`: `X` has shape `(N, window)`, `y` has shape `(N,)`, where
`X[k] = series[k : k+window]` and `y[k] = series[k + window + horizon − 1]`, and `N = len(series) − window − horizon + 1`.


In [8]:
def make_windows(series, window, horizon=1):
    N = len(series) - window - horizon + 1
    X = np.stack([series[k:k + window] for k in range(N)])
    y = np.array([series[k + window + horizon - 1] for k in range(N)])
    return X, y

In [9]:
# ▶ self-check
s = np.arange(10, 70, 10)
r = make_windows(s, 3, 1)
dl.check("3 windows of width 3", r is not None and r[0].shape == (3, 3))
dl.check("first window and target: [10,20,30] → 40", r is not None and list(r[0][0]) == [10, 20, 30] and r[1][0] == 40)
dl.check("last target is 60", r is not None and r[1][-1] == 60)
r2 = make_windows(s, 2, 3)
dl.check("horizon=3: [10,20] → 50 and count = 6−2−3+1 = 2", r2 is not None and len(r2[1]) == 2 and list(r2[0][0]) == [10, 20] and r2[1][0] == 50)

✅ 3 windows of width 3
✅ first window and target: [10,20,30] → 40
✅ last target is 60
✅ horizon=3: [10,20] → 50 and count = 6−2−3+1 = 2


---
### 🧩 Problem 11.5: Predict the shapes  &nbsp; `⭐`

Consider `lstm = nn.LSTM(10, 20, num_layers=2, batch_first=True, bidirectional=True)` and an input of shape `(8, 15, 10)`.
Without running it, fill the dictionary `shapes` with the shapes (tuples) of `out`, `h_n` and `c_n`. Then run it to verify.


In [10]:
shapes = {"out": (8, 15, 40), "h_n": (4, 8, 20), "c_n": (4, 8, 20)}

In [11]:
# ▶ self-check
lstm = nn.LSTM(10, 20, num_layers=2, batch_first=True, bidirectional=True)
o, (h, c) = lstm(torch.randn(8, 15, 10))
dl.check("out shape", shapes["out"] == tuple(o.shape))
dl.check("h_n shape", shapes["h_n"] == tuple(h.shape))
dl.check("c_n shape", shapes["c_n"] == tuple(c.shape))

✅ out shape
✅ h_n shape
✅ c_n shape
